## Output Parser

In [50]:
import os
import json
from dotenv import load_dotenv
load_dotenv()

API_KEY = os.getenv('AZURE_OPENAI_API_KEY')
BASE_URL = os.getenv('OPENAI_BASE_URL')

In [26]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model = 'gpt-5.1',
    api_key = API_KEY,
    base_url = BASE_URL,
    temperature = 1.8
)

In [27]:
# Without parser 
response = llm.invoke('What is the capital of France?')
print(response)
print(response.content)

content='The capital of France is Paris.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 13, 'total_tokens': 30, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 7, 'engine_ttft_ms': 73, 'engine_ttlt_ms': 192, 'pre_inference_ms': 83, 'service_tbt_ms': 7, 'service_ttft_ms': 272, 'service_ttlt_ms': 387, 'total_duration_ms': 306, 'user_visible_ttft_ms': 190}}, 'model_provider': 'openai', 'model_name': 'gpt-5.1-2025-11-13', 'system_fingerprint': None, 'id': 'chatcmpl-DdzLgfS3VAdOJGD7KE99vJATd7SGq', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019e1245-3956-7142-8170-fafdf8db3155-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 13, 'output_tokens': 17, 'total_tokens': 30, 'in

### String Output Parser

In [28]:
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()

In [29]:
response = llm.invoke('What is the capital of China?')
response = parser.invoke(response)

print(response)
print(type(response))

The capital of China is Beijing.
<class 'langchain_core.messages.base.TextAccessor'>


In [30]:
# Making a chain where it is really helpful

chain = llm | parser
response = chain.invoke('What is the capital of Germany?')
print(response)

The capital of Germany is Berlin.


In [31]:
# Using it with a prompt template
from langchain_core.prompts import ChatPromptTemplate
pro_con_template = ChatPromptTemplate.from_template(
    template = 'Give me 3 pros and cons of {topic}.'
)

In [32]:
pro_con_chain = pro_con_template | llm | parser

pro_con_chain.invoke({"topic": "artificial intelligence"})

'**3 Pros of Artificial Intelligence**\n\n1. **Efficiency and automation**  \n   - AI can handle repetitive, time-consuming tasks (data entry, scheduling, basic customer support) faster and with fewer errors than humans.\n\n2. **Improved decision-making**  \n   - AI can analyze huge amounts of data, find patterns, and support better decisions in areas like healthcare diagnosis, fraud detection, or supply chain optimization.\n\n3. **24/7 availability**  \n   - AI systems don’t get tired, so they can provide continuous services such as monitoring, chat support, and real-time translation around the clock.\n\n---\n\n**3 Cons of Artificial Intelligence**\n\n1. **Job displacement**  \n   - Automation can reduce the need for certain types of labor, especially repetitive or routine jobs, potentially leading to unemployment or the need for retraining.\n\n2. **Bias and unfairness**  \n   - If AI systems are trained on biased data, they can reinforce or amplify discrimination in hiring, lending, 

### Pydantic Parser

In [33]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List, Optional

In [34]:
class Person(BaseModel):
    name: str = Field(description='Name of the person')
    age: int = Field(description = 'Age of the person')
    city: str = Field(description = 'City where the person lives')

parser = PydanticOutputParser(pydantic_object=Person)

In [41]:
person_template = ChatPromptTemplate.from_template(
    template = 'Generate the name, age and city of a fictional {place} person, \n\nFormat Instruction: {foramt_instruction}',
    # input_variables = ['place'],
    partial_variables = {'foramt_instruction': parser.get_format_instructions()}
)

print(person_template.invoke({'place': 'Chinese'}))

messages=[HumanMessage(content='Generate the name, age and city of a fictional Chinese person, \n\nFormat Instruction: The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"name": {"description": "Name of the person", "title": "Name", "type": "string"}, "age": {"description": "Age of the person", "title": "Age", "type": "integer"}, "city": {"description": "City where the person lives", "title": "City", "type": "string"}}, "required": ["name", "age", "city"]}\n```', additional_kwargs={}, response_metadata={})]


In [43]:
person_chain  = person_template | llm | parser
response = person_chain.invoke({'place': 'Spain'})
response

Person(name='Lucía Rodríguez Martín', age=34, city='Sevilla')

In [56]:
print(response)
print(type(response))
# print(response.dict())
# print(response.name)
# print(response.dict()['name'])
print(response.name)

name='Lucía Rodríguez Martín' age=34 city='Sevilla'
<class '__main__.Person'>
Lucía Rodríguez Martín


### JSON Output Parser

In [57]:
from langchain_core.output_parsers import JsonOutputParser
json_parser = JsonOutputParser()

In [59]:
parser = JsonOutputParser()

pro_con_template = ChatPromptTemplate.from_template(
    template = 'Give me 3 pros and cons  and cons of {topic}. \n\n Format Instruction: {format_instruction}',
    partial_variables = {'format_instruction': parser.get_format_instructions()}
)

pro_con_chain = pro_con_template | llm | parser

In [61]:
response = pro_con_chain.invoke({"topic": "Agentic AI"})
print(response)
print(type(response))

{'pros': [{'title': 'Autonomous Task Execution', 'description': 'Agentic AI can plan and carry out multi-step tasks with minimal human intervention, increasing efficiency and freeing people to focus on higher-level work.'}, {'title': 'Improved Adaptability', 'description': 'By continuously observing outcomes and adjusting behavior, agentic systems can adapt to dynamic environments better than static, rule-based systems.'}, {'title': 'Scalable Decision-Making', 'description': 'Agentic AI can coordinate many decisions across systems or workflows (e.g., monitoring, scheduling, optimization), enabling scalable and persistent operation beyond human capacity.'}], 'cons': [{'title': 'Unintended Behavior and Misalignment', 'description': 'Autonomy increases the risk that the system pursues goals in unintended ways (e.g., optimizing a proxy metric at the expense of safety, ethics, or user intent).'}, {'title': 'Complexity of Oversight and Debugging', 'description': 'Because agentic AI makes its

In [64]:
person_schema = {
    "type": "object",
    "properties": {
        "name": {"type": "string"},
        "age": {"type": "integer"},
        "city": {"type": "string"}
    },
    "required": ["name", "age"]
}

person_parser = JsonOutputParser(schema=person_schema)

person_tempalte = ChatPromptTemplate.from_template(
    template = 'Generate the name, age and city of a fictional {place} person, \n\n Format Instruction: {format_instruction}',
    partial_variables = {'format_instruction': person_parser.get_format_instructions()}
)

person_chain = person_tempalte | llm | person_parser

In [65]:
response = person_chain.invoke({'place': 'French'})
print(response)
print(type(response))

{'name': 'Claire Dupont', 'age': 32, 'city': 'Lyon'}
<class 'dict'>


### Other Parsers
These are not being used today, these were used earlier and hence moved to classic module

#### Date time

In [67]:
from langchain_classic.output_parsers import DatetimeOutputParser
from datetime import datetime
parser = DatetimeOutputParser()

In [88]:
chain = llm | parser
dt = chain.invoke(
    "Output a random datetime in %Y-%m-%dT%H:%M:%S.%fZ. "
    # "Output a random datetim/e in DD-MM-YYYY HH:MM:SS"
    "Don't say anything else"
)

In [89]:
print(dt, type(dt))
print(dt.date())
print(dt.time())
print(dt.strftime('%d-%m-%Y %H:%M:%S.%fZ'))

2021-07-14 09:23:45.382917 <class 'datetime.datetime'>
2021-07-14
09:23:45.382917
14-07-2021 09:23:45.382917Z


In [103]:
dt = dt.replace(microsecond=0)
print(dt.strftime('%Y-%m-%dT%H:%M:%S.%fZ'))

2021-07-14T09:23:45.000000Z


#### Boolean

In [105]:
llm.invoke('"Are you an AI? YES or NO only"').content

'YES'

In [106]:
from langchain_classic.output_parsers import BooleanOutputParser
bool_parser = BooleanOutputParser()

In [107]:
chain = llm | bool_parser
response = chain.invoke('Give me random "YES" or "NO" answer')
print(response, type(response))

True <class 'bool'>
